In [0]:
from datetime import date
from delta.tables import DeltaTable

In [0]:

# 1. Read raw JSON from S3 (Bronze)
df_bronze = spark.read \
    .option("multiline", "true") \
    .json("s3a://aws-scraping-test-oks/raw/products/")

# 2. Clean and cast data types
df_silver = df_bronze \
    .withColumn("price", df_bronze["price"].cast("double")) \
    .withColumn("scraped_at", df_bronze["scraped_at"].cast("timestamp")) \
    .dropDuplicates(["sku", "scraped_at"])



In [0]:
# 3. Data quality checks --> Circuit Breaker
assert df_silver.filter("price IS NULL").count() == 0, "Знайдено рядки без ціни"
assert df_silver.count() == df_silver.select("sku", "scraped_at").distinct().count(), \
    "Знайдено дублікати (sku, scraped_at)"

In [0]:
# 4. Write to existing Delta table gold_db.silver_products using liquid clustering

from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "gold_db.silver_products")

target.alias("t").merge(
    df_silver.alias("s"),
    "t.sku = s.sku AND t.scraped_at = s.scraped_at"
).whenNotMatchedInsertAll().execute()

In [0]:
# 5. Data quality checks --> inserted rows>0 or there already rows with current date

inserted = result.collect()[0]["num_inserted_rows"]

assert inserted > 0 or spark.sql("SELECT MAX(date) FROM gold_db.silver_products").collect()[0][0] == date.today(), \
    "MERGE inserted 0 rows and no data exists for today — possible scraping failure"

In [0]:

# %sql
# select *
# from gold_db.silver_products